# PyTorch Distributed Training with Online Feature Store Serving

End-to-end MLOps workflow: Feature Store → Distributed GPU Training → Model Registry → SPCS Deployment → Online Inference → Monitoring.

**Flow:** Feature Store (online serving) → PyTorchDistributor (multi-GPU) → Custom Model wrapper → SPCS (GPU inference) → Model Monitor (drift + performance)

## 1. Setup

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

from snowflake.ml.runtime_cluster import scale_cluster
scale_cluster(3)

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader

from snowflake.ml.modeling.preprocessing import LabelEncoder, MinMaxScaler
from snowflake.ml.modeling.pipeline import Pipeline
from snowflake.ml.modeling.distributors.pytorch import (
    PyTorchDistributor,
    PyTorchScalingConfig,
    WorkerResourceConfig,
    get_context,
)
from snowflake.ml.data.sharded_data_connector import ShardedDataConnector
from snowflake.snowpark.context import get_active_session

import warnings
warnings.filterwarnings('ignore')

DB = "DYNAMIC_PRICING"
SCHEMA = "PRICING_MODEL"

session = get_active_session()
session.use_schema(f"{DB}.{SCHEMA}")

print(f"Database: {DB}")
print(f"Schema: {SCHEMA}")

## 1.5 Feature Store Setup

Register **Feature Views** with `online_enabled=True` — this enables low-latency point lookups at inference time. The CustomModel will call `retrieve_feature_values()` to fetch features on-demand.

In [ ]:
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode
from snowflake.ml.feature_store.feature_view import OnlineConfig

FS_SCHEMA = "FS_SCHEMA"

fs = FeatureStore(
    session=session,
    database=DB,
    name=FS_SCHEMA,
    default_warehouse=session.get_current_warehouse(),
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

product_entity = Entity(name="PRODUCT", join_keys=["PRODUCT_ID"])
store_entity = Entity(name="STORE", join_keys=["STORE_ID"])

fs.register_entity(product_entity)
fs.register_entity(store_entity)

print(f"Feature Store initialized: {DB}.{FS_SCHEMA}")
print(f"Entities registered: PRODUCT, STORE")

## 2. Load Features

In [ ]:
store_df = session.sql(f"""
    SELECT STORE_ID, REGION, CITY, STORE_TYPE, LAT, LON
    FROM {DB}.{SCHEMA}.DIM_STORE
    ORDER BY STORE_ID
""")
NUM_STORES = store_df.count()
print(f"Stores: {NUM_STORES}")
store_df.show(5)

In [ ]:
product_df = session.sql(f"""
    SELECT PRODUCT_ID, CATEGORY, SUBCATEGORY, BRAND, COST, BASE_PRICE, WEIGHT
    FROM {DB}.{SCHEMA}.DIM_PRODUCT
    ORDER BY PRODUCT_ID
""")
NUM_PRODUCTS = product_df.count()
print(f"Products: {NUM_PRODUCTS}")
product_df.show(5)

In [ ]:
product_fv = FeatureView(
    name="PRODUCT_FEATURES",
    entities=[product_entity],
    feature_df=product_df,
    desc="Product attributes for dynamic pricing",
    refresh_freq='10 minutes',
    online_config=OnlineConfig(enable=True)
)
product_fv = fs.register_feature_view(
    feature_view=product_fv,
    version="V1",
    block=True,
    overwrite=True,
)
print(f"Registered: {product_fv.name} version {product_fv.version}")

In [ ]:
store_fv = FeatureView(
    name="STORE_FEATURES",
    entities=[store_entity],
    feature_df=store_df,
    desc="Store attributes for dynamic pricing",
    refresh_freq='10 minutes',
    online_config=OnlineConfig(enable=True)
)
store_fv = fs.register_feature_view(
    feature_view=store_fv,
    version="V1",
    block=True,
    overwrite=True,
)
print(f"Registered: {store_fv.name} version {store_fv.version}")

## 3. Prepare Training Data

Build a spine of entity IDs, join features from the Feature Store using `generate_dataset()`, then fit a preprocessing pipeline. **Pipeline params are serialized to JSON** — this artifact travels with the model to ensure training-serving consistency.

In [ ]:
spine_df = session.sql(f"""
    SELECT PRODUCT_ID, STORE_ID, ROUND(AVG(SELLING_PRICE), 2) AS TARGET_PRICE
    FROM {DB}.{SCHEMA}.FACT_SALES_DAILY
    GROUP BY PRODUCT_ID, STORE_ID
""")
print(f"Spine records (product-store pairs): {spine_df.count()}")

training_dataset = fs.generate_dataset(
    name="TT_TRAINING_DATA",
    spine_df=spine_df,
    features=[product_fv, store_fv],
    output_type="dataset",
)

training_df = training_dataset.read.to_snowpark_dataframe()
print(f"Training samples from Feature Store: {training_df.count()}")
training_df.show(5)

In [ ]:
PRODUCT_CAT_COLS = ["CATEGORY", "SUBCATEGORY", "BRAND"]
PRODUCT_NUM_COLS = ["COST", "BASE_PRICE", "WEIGHT"]
STORE_CAT_COLS = ["REGION", "CITY", "STORE_TYPE"]
STORE_NUM_COLS = ["LAT", "LON"]

pipeline_steps = []
for col in PRODUCT_CAT_COLS + STORE_CAT_COLS:
    pipeline_steps.append((f"LE_{col}", LabelEncoder(input_cols=[col], output_cols=[col])))

pipeline_steps.append(("MMS", MinMaxScaler(
    input_cols=PRODUCT_NUM_COLS + STORE_NUM_COLS,
    output_cols=PRODUCT_NUM_COLS + STORE_NUM_COLS,
)))

pipeline = Pipeline(steps=pipeline_steps)
processed_df = pipeline.fit(training_df).transform(training_df)

import json

pipeline_params = {"label_encoders": {}, "scalers": {}}
raw_pd = training_df.select(PRODUCT_CAT_COLS + STORE_CAT_COLS + PRODUCT_NUM_COLS + STORE_NUM_COLS).to_pandas()

for col in PRODUCT_CAT_COLS + STORE_CAT_COLS:
    pipeline_params["label_encoders"][col] = sorted(raw_pd[col].dropna().unique().tolist())

for col in PRODUCT_NUM_COLS + STORE_NUM_COLS:
    pipeline_params["scalers"][col] = {"min": float(raw_pd[col].min()), "max": float(raw_pd[col].max())}

pipeline_params_path = "/tmp/pipeline_params.json"
with open(pipeline_params_path, "w") as f:
    json.dump(pipeline_params, f)

print(f"Processed dataset: {processed_df.count()} rows")
print(f"Pipeline params saved to {pipeline_params_path}")
processed_df.show(5)

In [ ]:
train_df, val_df = processed_df.random_split([0.8, 0.2], seed=42)
train_df.write.mode("overwrite").save_as_table(f"{DB}.{SCHEMA}.TT_TRAIN_DATA", table_type="temporary")
val_df.write.mode("overwrite").save_as_table(f"{DB}.{SCHEMA}.TT_VAL_DATA", table_type="temporary")

print(f"Train: {train_df.count()} | Val: {val_df.count()}")

## 4. Model Definition

Two-Tower architecture: separate embedding + MLP towers for each entity type, combined head for prediction. Model is defined **inside** `train_func()` — `PyTorchDistributor` serializes and ships the entire function to workers.

In [ ]:
EMBEDDING_DIM = 32
HIDDEN_DIM = 64
NUM_PRODUCT_FEATURES = len(PRODUCT_CAT_COLS) + len(PRODUCT_NUM_COLS)  # 6
NUM_STORE_FEATURES = len(STORE_CAT_COLS) + len(STORE_NUM_COLS)  # 5

print(f"Embedding dim: {EMBEDDING_DIM}")
print(f"Hidden dim: {HIDDEN_DIM}")
print(f"Product features: {NUM_PRODUCT_FEATURES}")
print(f"Store features: {NUM_STORE_FEATURES}")

## 5. Distributed Training

`PyTorchDistributor` handles multi-GPU distribution automatically. `ShardedDataConnector` partitions Snowflake data across workers; `DataLoader` then batches each worker's partition. Model weights are saved to a stage path for the CustomModel to load.

In [ ]:
NUM_EPOCHS = 10
BATCH_SIZE = 256
LEARNING_RATE = 0.001

print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")

In [ ]:
def train_func():
    """Distributed training function for Two-Tower model."""
    import os
    import time
    import torch
    import torch.nn as nn
    import torch.distributed as dist
    from torch.nn.parallel import DistributedDataParallel as DDP
    from torch.utils.data import DataLoader
    from snowflake.ml.modeling.distributors.pytorch import get_context
    
    context = get_context()
    rank = context.get_rank()
    world_size = context.get_world_size()
    model_dir = context.get_model_dir()
    local_rank = int(os.environ.get("LOCAL_RANK", rank % torch.cuda.device_count()))
    
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.manual_seed(42)
    torch.cuda.set_device(local_rank)
    device = torch.device(f"cuda:{local_rank}")
    
    class _TwoTowerModel(nn.Module):
        def __init__(self, num_products, num_stores, num_p_feat, num_s_feat, emb_dim=32, hidden=64):
            super().__init__()
            self.product_embedding = nn.Embedding(num_products, emb_dim)
            self.product_tower = nn.Sequential(
                nn.Linear(emb_dim + num_p_feat, hidden), nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(hidden, hidden // 2), nn.ReLU(),
            )
            self.store_embedding = nn.Embedding(num_stores, emb_dim)
            self.store_tower = nn.Sequential(
                nn.Linear(emb_dim + num_s_feat, hidden), nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(hidden, hidden // 2), nn.ReLU(),
            )
            self.head = nn.Sequential(
                nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(hidden // 2, 1),
            )
        
        def forward(self, product_ids, store_ids, product_features, store_features):
            p_emb = self.product_embedding(product_ids)
            p_repr = self.product_tower(torch.cat([p_emb, product_features], dim=-1))
            s_emb = self.store_embedding(store_ids)
            s_repr = self.store_tower(torch.cat([s_emb, store_features], dim=-1))
            return self.head(torch.cat([p_repr, s_repr], dim=-1))
    
    model = _TwoTowerModel(
        num_products=NUM_PRODUCTS,
        num_stores=NUM_STORES,
        num_p_feat=NUM_PRODUCT_FEATURES,
        num_s_feat=NUM_STORE_FEATURES,
        emb_dim=EMBEDDING_DIM,
        hidden=HIDDEN_DIM,
    ).to(device)
    ddp_model = DDP(model, device_ids=[local_rank])
    
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(ddp_model.parameters(), lr=LEARNING_RATE)
    
    dataset_map = context.get_dataset_map()
    train_pipe = dataset_map["train"].get_shard().to_torch_datapipe(batch_size=BATCH_SIZE, shuffle=True)
    val_pipe = dataset_map["val"].get_shard().to_torch_datapipe(batch_size=BATCH_SIZE, shuffle=False)
    train_loader = DataLoader(train_pipe, batch_size=None)
    val_loader = DataLoader(val_pipe, batch_size=None)
    
    product_feat_cols = PRODUCT_CAT_COLS + PRODUCT_NUM_COLS
    store_feat_cols = STORE_CAT_COLS + STORE_NUM_COLS
    
    for epoch in range(NUM_EPOCHS):
        ddp_model.train()
        start = time.time()
        train_loss, n_batches = 0.0, 0
        
        for batch in train_loader:
            product_ids = (batch["PRODUCT_ID"].squeeze().long() - 1).to(device)
            store_ids = (batch["STORE_ID"].squeeze().long() - 1).to(device)
            product_feats = torch.stack([batch[c].squeeze().float() for c in product_feat_cols], dim=-1).to(device)
            store_feats = torch.stack([batch[c].squeeze().float() for c in store_feat_cols], dim=-1).to(device)
            target = batch["TARGET_PRICE"].squeeze().float().to(device)
            
            optimizer.zero_grad()
            pred = ddp_model(product_ids, store_ids, product_feats, store_feats).squeeze()
            loss = criterion(pred, target)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            n_batches += 1
        
        avg_train = torch.tensor(train_loss / max(n_batches, 1), device=device)
        dist.all_reduce(avg_train)
        avg_train = avg_train.item() / world_size
        
        ddp_model.eval()
        val_loss, val_batches = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                product_ids = (batch["PRODUCT_ID"].squeeze().long() - 1).to(device)
                store_ids = (batch["STORE_ID"].squeeze().long() - 1).to(device)
                product_feats = torch.stack([batch[c].squeeze().float() for c in product_feat_cols], dim=-1).to(device)
                store_feats = torch.stack([batch[c].squeeze().float() for c in store_feat_cols], dim=-1).to(device)
                target = batch["TARGET_PRICE"].squeeze().float().to(device)
                pred = ddp_model(product_ids, store_ids, product_feats, store_feats).squeeze()
                val_loss += criterion(pred, target).item()
                val_batches += 1
        
        avg_val = torch.tensor(val_loss / max(val_batches, 1), device=device)
        dist.all_reduce(avg_val)
        avg_val = avg_val.item() / world_size
        
        dist.barrier()
        
        if rank == 0:
            print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train: {avg_train:.4f} | Val: {avg_val:.4f} | Time: {time.time()-start:.1f}s")
    
    if rank == 0:
        torch.save(model.state_dict(), os.path.join(model_dir, "two_tower.pth"))
        print(f"Model saved to {model_dir}/two_tower.pth")
    
    dist.destroy_process_group()

In [ ]:
trainer = PyTorchDistributor(
    train_func=train_func,
    scaling_config=PyTorchScalingConfig(
        num_nodes=3,
        num_workers_per_node=1,
        resource_requirements_per_worker=WorkerResourceConfig(num_cpus=0, num_gpus=1),
    ),
)

train_data = session.table(f"{DB}.{SCHEMA}.TT_TRAIN_DATA")
val_data = session.table(f"{DB}.{SCHEMA}.TT_VAL_DATA")

print(f"Training: {train_data.count()} rows")
print(f"Validation: {val_data.count()} rows")
print("Launching distributed training...")

out = trainer.run(
    dataset_map=dict(
        train=ShardedDataConnector.from_dataframe(train_data),
        val=ShardedDataConnector.from_dataframe(val_data),
    )
)
print("Training complete.")

## 6. Model Registry and SPCS Deployment

PyTorch models require a **CustomModel wrapper** — this bundles the model class, weights, Feature Store client, and preprocessing logic into a single deployable unit. `Session.builder.getOrCreate()` inside SPCS picks up the container's session automatically. The model is deployed to a GPU compute pool with a public HTTPS endpoint (`ingress_enabled=True`).

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model import custom_model

class TwoTowerInferenceModel(custom_model.CustomModel):
    
    def __init__(self, context: custom_model.ModelContext) -> None:
        super().__init__(context)
        import json
        import torch
        import torch.nn as nn
        
        class _TwoTowerModel(nn.Module):
            def __init__(self, num_products, num_stores, num_p_feat, num_s_feat, emb_dim=32, hidden=64):
                super().__init__()
                self.product_embedding = nn.Embedding(num_products, emb_dim)
                self.product_tower = nn.Sequential(
                    nn.Linear(emb_dim + num_p_feat, hidden), nn.ReLU(), nn.Dropout(0.2),
                    nn.Linear(hidden, hidden // 2), nn.ReLU(),
                )
                self.store_embedding = nn.Embedding(num_stores, emb_dim)
                self.store_tower = nn.Sequential(
                    nn.Linear(emb_dim + num_s_feat, hidden), nn.ReLU(), nn.Dropout(0.2),
                    nn.Linear(hidden, hidden // 2), nn.ReLU(),
                )
                self.head = nn.Sequential(
                    nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(0.2),
                    nn.Linear(hidden // 2, 1),
                )
            
            def forward(self, product_ids, store_ids, product_features, store_features):
                p_emb = self.product_embedding(product_ids)
                p_repr = self.product_tower(torch.cat([p_emb, product_features], dim=-1))
                s_emb = self.store_embedding(store_ids)
                s_repr = self.store_tower(torch.cat([s_emb, store_features], dim=-1))
                return self.head(torch.cat([p_repr, s_repr], dim=-1))
        
        self.model = _TwoTowerModel(200, 50, 6, 5, emb_dim=32, hidden=64)
        self.model.load_state_dict(torch.load(context.path("two_tower.pth"), map_location="cpu"))
        self.model.eval()
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        
        with open(context.path("pipeline_params.json")) as f:
            self.pipeline_params = json.load(f)
        
        self.product_cat = ["CATEGORY", "SUBCATEGORY", "BRAND"]
        self.product_num = ["COST", "BASE_PRICE", "WEIGHT"]
        self.store_cat = ["REGION", "CITY", "STORE_TYPE"]
        self.store_num = ["LAT", "LON"]
    
    def _preprocess(self, features):
        for col, classes in self.pipeline_params["label_encoders"].items():
            mapping = {v: i for i, v in enumerate(classes)}
            features[col] = features[col].map(mapping).fillna(-1).astype(int).clip(lower=0)
        for col, params in self.pipeline_params["scalers"].items():
            features[col] = features[col].fillna(0)
            rng = params["max"] - params["min"]
            if rng > 0:
                features[col] = (features[col] - params["min"]) / rng
        return features
    
    @custom_model.inference_api
    def predict(self, input_df: pd.DataFrame) -> pd.DataFrame:
        import os
        import torch
        import numpy as np
        from snowflake.snowpark import Session
        from snowflake.ml.feature_store import FeatureStore

        with open("/snowflake/session/token", "r") as f:
            token = f.read().strip()

        session = Session.builder.configs({
            "account": os.environ["SNOWFLAKE_ACCOUNT"],
            "host": os.environ["SNOWFLAKE_HOST"],
            "authenticator": "oauth",
            "token": token,
            "warehouse": "COMPUTE_WH",
            "database": "DYNAMIC_PRICING",
            "schema": "PRICING_MODEL",
        }).create()

        try:
            fs = FeatureStore(
                session=session,
                database="DYNAMIC_PRICING",
                name="FS_SCHEMA",
                default_warehouse=session.get_current_warehouse(),
            )
            product_fv = fs.get_feature_view("PRODUCT_FEATURES", "V1")
            store_fv = fs.get_feature_view("STORE_FEATURES", "V1")

            spine = session.create_dataframe(input_df[["PRODUCT_ID", "STORE_ID"]])

            features = fs.retrieve_feature_values(
                spine_df=spine,
                features=[product_fv, store_fv],
            ).to_pandas()

            missing = features[self.product_cat + self.store_cat].isna().any(axis=1)
            if missing.any():
                print(f"Warning: {missing.sum()} rows missing features (unknown IDs)")

            features = self._preprocess(features)

            p_ids = torch.tensor(features["PRODUCT_ID"].values - 1, dtype=torch.long).to(self.device)
            s_ids = torch.tensor(features["STORE_ID"].values - 1, dtype=torch.long).to(self.device)
            p_feats = torch.tensor(features[self.product_cat + self.product_num].values, dtype=torch.float32).to(self.device)
            s_feats = torch.tensor(features[self.store_cat + self.store_num].values, dtype=torch.float32).to(self.device)

            with torch.no_grad():
                predictions = self.model(p_ids, s_ids, p_feats, s_feats).squeeze(-1).cpu().numpy()

            predictions = np.atleast_1d(predictions)
            result = pd.DataFrame({"PREDICTED_PRICE": predictions})
            result.loc[missing.values, "PREDICTED_PRICE"] = None
            return result
        finally:
            session.close()

stage_model_path = out.get_model_dir() + "/two_tower.pth"
local_model_path = "/tmp/two_tower.pth"
session.file.get(stage_model_path, "/tmp")

model_context = custom_model.ModelContext(
    artifacts={
        "two_tower.pth": local_model_path,
        "pipeline_params.json": pipeline_params_path,
    }
)
inference_model = TwoTowerInferenceModel(model_context)

sample_input = pd.DataFrame({"PRODUCT_ID": [1, 2, 3], "STORE_ID": [1, 2, 3]})
sample_output = inference_model.predict(sample_input)
print("Sample predictions:")
print(sample_output)

In [ ]:
reg = Registry(session=session, database_name=DB, schema_name=SCHEMA)

mv = reg.log_model(
    inference_model,
    model_name="TWO_TOWER_PRICING",
    version_name="V1",
    sample_input_data=sample_input,
    pip_requirements=["torch>=2.0.0", "numpy", "pandas", "snowflake-snowpark-python"],
    target_platforms=["SNOWPARK_CONTAINER_SERVICES"],
    options={"cuda_version": "12.3"},
    comment="Two-Tower dynamic pricing model with Feature Store online serving",
)
print(f"Registered: {mv.model_name} version {mv.version_name}")

session.sql(f"DROP SERVICE IF EXISTS {DB}.{SCHEMA}.TWO_TOWER_INFERENCE").collect()

mv.create_service(
    service_name="TWO_TOWER_INFERENCE",
    service_compute_pool="SYSTEM_COMPUTE_POOL_GPU",
    image_build_compute_pool="SYSTEM_COMPUTE_POOL_CPU",
    ingress_enabled=True,
    gpu_requests=1,
    num_workers=1,
    max_instances=1,
    autocapture=True
)

print("Service creation initiated (5-15 minutes)")
print(mv.list_services())

## 7. Test Inference

In [ ]:
%%sql -r dataframe_2
SHOW SERVICES LIKE 'TWO_TOWER_INFERENCE' IN SCHEMA DYNAMIC_PRICING.PRICING_MODEL;

In [ ]:
test_df = pd.DataFrame({
    "PRODUCT_ID": [1, 50, 100, 200],
    "STORE_ID": [1, 25, 50, 10],
})

result = mv.run(test_df, function_name="predict", service_name="TWO_TOWER_INFERENCE")
print("Inference results:")
print(result)

## 8. Secure Online Inference Endpoint

SPCS with `ingress_enabled=True` exposes a **public HTTPS endpoint** — any app can call it without the Snowflake SDK. **Authentication**: Programmatic Access Tokens (PATs) secure the endpoint via the `Authorization` header.

In [ ]:
# Discover the public endpoint URL
services_df = mv.list_services()
print(services_df)

endpoint_url = services_df.iloc[0]["inference_endpoint"]
predict_url = f"https://{endpoint_url}/predict"
print(f"\nPublic inference URL: {predict_url}")

In [ ]:
import requests
import json
import time

# ── Replace with your PAT (or set as environment variable) ──
# Generate a PAT in Snowsight: User Menu > My Profile > Programmatic Access Tokens
# Or via SQL: ALTER USER <username> ADD PROGRAMMATIC ACCESS TOKEN pat_demo;
PAT_TOKEN = os.environ.get("SNOWFLAKE_PAT", "<YOUR_PAT_HERE>")

headers = {
    "Authorization": f'Snowflake Token="{PAT_TOKEN}"',
    "Content-Type": "application/json",
}

payload = {
    "dataframe_records": [
        {"PRODUCT_ID": 1, "STORE_ID": 1},
        {"PRODUCT_ID": 50, "STORE_ID": 25},
        {"PRODUCT_ID": 100, "STORE_ID": 50},
        {"PRODUCT_ID": 200, "STORE_ID": 10},
    ]
}

response = requests.post(predict_url, headers=headers, json=payload, timeout=30)

print(f"Status: {response.status_code}")
print(f"Response:\n{json.dumps(response.json(), indent=2)}")

**Step 1:** Export your PAT as an environment variable:
```bash
export SNOWFLAKE_PAT='your-pat-here'
```

**Step 2:** Copy the `predict_url` printed above and run:
```bash
curl -X POST "<YOUR_PREDICT_URL>" \
  -H 'Authorization: Snowflake Token="'"$SNOWFLAKE_PAT"'"' \
  -H 'Content-Type: application/json' \
  -d '{
    "dataframe_records": [
      {"PRODUCT_ID": 1, "STORE_ID": 1},
      {"PRODUCT_ID": 50, "STORE_ID": 25},
      {"PRODUCT_ID": 150, "STORE_ID": 30}
    ]
  }'
```

Replace `<YOUR_PREDICT_URL>` with the URL from the cell above.

## 9. Simulate Production Inference Traffic

To demonstrate ML monitoring, we need a history of inference requests. Instead of waiting days for real traffic, we **backfill an inference log** with simulated data spanning 30 days.

**Strategy:**
- **Days 1-15 (stable)**: Normal product/store mix. Predictions closely match actuals.
- **Days 16-30 (drift)**: Skew toward electronics, inflate costs by 30%. Predictions diverge from actuals.

The Model Monitor reads from this table and computes drift/performance metrics per day.

In [ ]:
# Create the inference log table (source for the monitor)
session.sql(f"""
CREATE OR REPLACE TABLE {DB}.{SCHEMA}.INFERENCE_LOG (
    ID NUMBER AUTOINCREMENT,
    TIMESTAMP TIMESTAMP_NTZ,
    PRODUCT_ID NUMBER,
    STORE_ID NUMBER,
    CATEGORY VARCHAR,
    COST NUMBER(10,2),
    BASE_PRICE NUMBER(10,2),
    WEIGHT NUMBER(10,2),
    PREDICTED_PRICE NUMBER(10,4),
    ACTUAL_PRICE NUMBER(10,4)
)
""").collect()

# Create baseline table from training data distribution
session.sql(f"""
CREATE OR REPLACE TABLE {DB}.{SCHEMA}.MONITORING_BASELINE AS
SELECT
    p.PRODUCT_ID,
    s.STORE_ID,
    p.CATEGORY,
    p.COST,
    p.BASE_PRICE,
    p.WEIGHT,
    ROUND(AVG(f.SELLING_PRICE), 4) AS PREDICTED_PRICE,
    ROUND(AVG(f.SELLING_PRICE), 4) AS ACTUAL_PRICE
FROM {DB}.{SCHEMA}.FACT_SALES_DAILY f
JOIN {DB}.{SCHEMA}.DIM_PRODUCT p ON f.PRODUCT_ID = p.PRODUCT_ID
JOIN {DB}.{SCHEMA}.DIM_STORE s ON f.STORE_ID = s.STORE_ID
GROUP BY p.PRODUCT_ID, s.STORE_ID, p.CATEGORY, p.COST, p.BASE_PRICE, p.WEIGHT
""").collect()

print("INFERENCE_LOG table created")
print(f"MONITORING_BASELINE rows: {session.table(f'{DB}.{SCHEMA}.MONITORING_BASELINE').count()}")

In [ ]:
import datetime

products_pd = session.sql(f"""
    SELECT PRODUCT_ID, CATEGORY, COST, BASE_PRICE, WEIGHT
    FROM {DB}.{SCHEMA}.DIM_PRODUCT
""").to_pandas()

stores_pd = session.sql(f"""
    SELECT STORE_ID FROM {DB}.{SCHEMA}.DIM_STORE
""").to_pandas()

actuals_pd = session.sql(f"""
    SELECT PRODUCT_ID, STORE_ID, ROUND(AVG(SELLING_PRICE), 4) AS AVG_PRICE
    FROM {DB}.{SCHEMA}.FACT_SALES_DAILY
    GROUP BY PRODUCT_ID, STORE_ID
""").to_pandas()
actuals_lookup = actuals_pd.set_index(["PRODUCT_ID", "STORE_ID"])["AVG_PRICE"].to_dict()

np.random.seed(42)
rows = []
base_date = datetime.datetime.now() - datetime.timedelta(days=30)

for day in range(30):
    ts = base_date + datetime.timedelta(days=day)
    is_drift = day >= 15

    if is_drift:
        n_electronics = 140
        n_other = 60
        elec_products = products_pd[products_pd["CATEGORY"] == "Electronics"].sample(
            n=n_electronics, replace=True, random_state=day
        )
        other_products = products_pd[products_pd["CATEGORY"] != "Electronics"].sample(
            n=n_other, replace=True, random_state=day
        )
        sampled_products = pd.concat([elec_products, other_products], ignore_index=True)
    else:
        sampled_products = products_pd.sample(n=200, replace=True, random_state=day)

    sampled_stores = stores_pd.sample(n=200, replace=True, random_state=day + 100)

    for i in range(200):
        pid = int(sampled_products.iloc[i]["PRODUCT_ID"])
        sid = int(sampled_stores.iloc[i]["STORE_ID"])
        category = sampled_products.iloc[i]["CATEGORY"]
        cost = float(sampled_products.iloc[i]["COST"])
        base_price = float(sampled_products.iloc[i]["BASE_PRICE"])
        weight = float(sampled_products.iloc[i]["WEIGHT"])

        actual = actuals_lookup.get((pid, sid), base_price * 1.1)

        if is_drift:
            cost = round(cost * 1.3, 2)
            base_price = round(base_price * 1.3, 2)
            predicted = round(actual * np.random.uniform(1.15, 1.45), 4)
            actual = round(actual * np.random.uniform(0.95, 1.05), 4)
        else:
            predicted = round(actual * np.random.uniform(0.97, 1.03), 4)
            actual = round(actual * np.random.uniform(0.98, 1.02), 4)

        rows.append({
            "TIMESTAMP": ts + datetime.timedelta(minutes=np.random.randint(0, 1440)),
            "PRODUCT_ID": pid,
            "STORE_ID": sid,
            "CATEGORY": category,
            "COST": cost,
            "BASE_PRICE": base_price,
            "WEIGHT": weight,
            "PREDICTED_PRICE": predicted,
            "ACTUAL_PRICE": actual,
        })

inference_df = pd.DataFrame(rows)
print(f"Generated {len(inference_df)} simulated inference records")
print(f"Stable period (days 1-15): {len(inference_df[inference_df['TIMESTAMP'] < base_date + datetime.timedelta(days=15)])} rows")
print(f"Drift period  (days 16-30): {len(inference_df[inference_df['TIMESTAMP'] >= base_date + datetime.timedelta(days=15)])} rows")

sp_df = session.create_dataframe(inference_df)
sp_df.write.mode("append").save_as_table(
    f"{DB}.{SCHEMA}.INFERENCE_LOG",
    column_order="name"
)

print(f"\nInserted into {DB}.{SCHEMA}.INFERENCE_LOG")
print(f"Total rows: {session.table(f'{DB}.{SCHEMA}.INFERENCE_LOG').count()}")

## 10. ML Observability & Monitoring

**Model Monitor** tracks drift (PSI) and performance (RMSE/MAE) over time. It compares inference data against a training baseline, supports segmented analysis (e.g., by category), and surfaces metrics in Snowsight's ML Monitoring dashboard.

- **Drift**: PSI > 0.25 indicates significant feature distribution shift
- **Performance**: Compares predicted vs actual prices from the inference log
- **Source**: `PRICING_INFERENCE_LOG` (populated by your inference pipeline)

In [ ]:
wh = session.get_current_warehouse().replace('"', '')

session.sql(f"USE SCHEMA {DB}.{SCHEMA}").collect()

session.sql(f"""
CREATE OR REPLACE MODEL MONITOR {DB}.{SCHEMA}.PRICING_MONITOR WITH
    MODEL = TWO_TOWER_PRICING
    VERSION = 'V1'
    FUNCTION = 'predict'
    SOURCE = {DB}.{SCHEMA}.INFERENCE_LOG
    BASELINE = {DB}.{SCHEMA}.MONITORING_BASELINE
    WAREHOUSE = {wh}
    REFRESH_INTERVAL = '1 day'
    AGGREGATION_WINDOW = '1 day'
    TIMESTAMP_COLUMN = TIMESTAMP
    ID_COLUMNS = ('ID')
    PREDICTION_SCORE_COLUMNS = ('PREDICTED_PRICE')
    ACTUAL_SCORE_COLUMNS = ('ACTUAL_PRICE')
    SEGMENT_COLUMNS = ('CATEGORY')
""").collect()

print("Model Monitor created: PRICING_MONITOR")
print(f"  Source: {DB}.{SCHEMA}.INFERENCE_LOG")
print(f"  Baseline: {DB}.{SCHEMA}.MONITORING_BASELINE")
print(f"  Segments: CATEGORY")

session.sql(f"SHOW MODEL MONITORS IN SCHEMA {DB}.{SCHEMA}").show()

In [ ]:
drift_results = session.sql(f"""
SELECT *
FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
    '{DB}.{SCHEMA}.PRICING_MONITOR',
    'POPULATION_STABILITY_INDEX',
    'COST',
    '1 DAY',
    DATEADD('DAY', -30, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
    CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
))
ORDER BY 1
""")

print("=== Drift Metric (PSI) for COST feature ===")
print("Higher values = more drift from baseline")
print("PSI > 0.1 = moderate drift | PSI > 0.25 = significant drift\n")
drift_results.show(30)

In [ ]:
perf_results = session.sql(f"""
SELECT *
FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC(
    '{DB}.{SCHEMA}.PRICING_MONITOR',
    'RMSE',
    '1 DAY',
    DATEADD('DAY', -30, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
    CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
))
ORDER BY 1
""")

print("=== Performance Metric (RMSE) ===")
print("Rising RMSE indicates model predictions are diverging from actuals\n")
perf_results.show(30)

mae_results = session.sql(f"""
SELECT *
FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC(
    '{DB}.{SCHEMA}.PRICING_MONITOR',
    'MAE',
    '1 DAY',
    DATEADD('DAY', -30, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
    CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
))
ORDER BY 1
""")

print("\n=== Performance Metric (MAE) ===")
mae_results.show(30)

In [ ]:
elec_drift = session.sql(f"""
SELECT *
FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
    '{DB}.{SCHEMA}.PRICING_MONITOR',
    'POPULATION_STABILITY_INDEX',
    'COST',
    '1 DAY',
    DATEADD('DAY', -30, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
    CURRENT_TIMESTAMP()::TIMESTAMP_NTZ,
    '{{"SEGMENTS": [{{"column": "CATEGORY", "value": "Electronics"}}]}}'
))
ORDER BY 1
""")

print("=== Drift for Electronics Segment ===")
print("Electronics was deliberately over-represented in the drift period\n")
elec_drift.show(30)

In [ ]:
%%sql -r dataframe_3
SELECT * FROM TABLE(INFERENCE_TABLE('DYNAMIC_PRICING.PRICING_MODEL.TWO_TOWER_PRICING'));